# Prueba de OOM con batch size 8 en todos los preprocesados

Notebook de estrés para comprobar qué combinaciones de preprocesado y canales pueden entrenarse con **batch size 8** sin provocar `CUDA out of memory`.

Configuración de la prueba:
- todos los preprocesados disponibles mostrados en el servidor
- los 6 experimentos de canales/máscaras del notebook preliminar
- 10 épocas completas por combinación
- AMP activado
- captura de OOM sin detener el barrido completo
- limpieza de memoria CUDA entre ejecuciones
- registro incremental de estado y pico de memoria GPU

El resultado principal queda en:
- `summary_all_experiments.csv`: una fila por combinación
- `oom_matrix.csv`: matriz preprocesado × experimento con `OK`, `OOM` o `ERROR`
- `summary_by_preprocessing.csv`: resumen agregado por preprocesado

Este notebook debe ejecutarse en el servidor que contiene los volúmenes y la GPU; aquí solo se prepara el barrido.


## Diseño del barrido

Endpoint multietiqueta:
- `Hemorragia`
- `Neumotórax`
- `Sin_complicacion` derivada

Se excluyen por defecto los pacientes cuya única complicación está fuera del endpoint principal, como `Derrame pleural`.

Se ejecutan 16 preprocesados × 6 configuraciones de entrada = **96 entrenamientos**. Cada entrenamiento intenta completar exactamente 10 épocas con `batch_size=8`.

La prueba registra por combinación:
- estado final: `OK`, `OOM` o `ERROR`
- etapa, época y lote en el que ocurrió el fallo
- épocas completadas
- memoria CUDA máxima asignada y reservada
- métricas parciales o finales cuando existan

Los preprocesados `multiwindowing_separadas` se cargan conservando sus canales 4D, en lugar de descartar ventanas.


In [ ]:
from pathlib import Path
import gc
import json
import math
import random
import time
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    hamming_loss,
    jaccard_score,
)

from monai.networks.nets import resnet18
from monai.transforms import Compose, RandFlip, RandRotate90, RandAffine, EnsureType

print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM total: {props.total_memory / 1024**3:.2f} GiB")


In [ ]:
PROJECT_ROOT = Path("/mnt/homeGPU/mcribilles/tfm")
CLINICAL_CSV = PROJECT_ROOT / "clinical_data/210pacientes/clinical_data.csv"
PREPROC_ROOT = PROJECT_ROOT / "volumenes_preprocesados"
NOTEBOOK_OUTPUT_ROOT = PROJECT_ROOT / "codigo/DL_multietiqueta/outputs_oom_bs8"
NOTEBOOK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_LABELS = ["Hemorragia", "Neumotórax", "Sin_complicacion"]
DROP_OFF_TARGET_ONLY = True

PREPROCESSINGS = [
    "resize_cube64",
    "resize_cube64_hu_m300_1400",
    "resize_cube64_hu_m600_1500",
    "resize_cube64_multiwindowing_separadas",
    "resize_cube128",
    "resize_cube128_hu_m300_1400",
    "resize_cube128_hu_m600_1500",
    "resize_cube128_multiwindowing_separadas",
    "resize_medium",
    "resize_medium_hu_m300_1400",
    "resize_medium_hu_m600_1500",
    "resize_medium_multiwindowing_separadas",
    "resize_small",
    "resize_small_hu_m300_1400",
    "resize_small_hu_m600_1500",
    "resize_small_multiwindowing_separadas",
]

EXPERIMENTS = [
    {
        "name": "lung_only_masked_ct",
        "channels": ["ct_lung"],
        "description": "Solo imagen dentro del pulmón",
    },
    {
        "name": "nodule_only_masked_ct",
        "channels": ["ct_nodule"],
        "description": "Solo imagen dentro del nódulo",
    },
    {
        "name": "ct_lung_nodule",
        "channels": ["ct_lung", "lung", "nodule"],
        "description": "Imagen pulmonar + máscara pulmón + máscara nódulo",
    },
    {
        "name": "nodule_vessels",
        "channels": ["ct_nodule", "nodule", "vessels"],
        "description": "Imagen del nódulo + máscara nódulo + máscara vasos",
    },
    {
        "name": "ct_lung_nodule_vessels",
        "channels": ["ct_lung", "lung", "nodule", "vessels"],
        "description": "Imagen pulmonar + pulmón + nódulo + vasos",
    },
    {
        "name": "weighted_lung_nodule_vessels",
        "channels": ["weighted_ct"],
        "weights": {"lung": 0.25, "nodule": 1.00, "vessels": 0.75},
        "description": "Canal/ventanas de CT ponderadas por pulmón-nódulo-vasos",
    },
]

REQUIRE_COMMON_COHORT = True
MAX_EXPERIMENTS = None
MAX_PREPROCS = None

TRAIN_SIZE = 0.80
RANDOM_SEED = 42
BATCH_SIZE = 8
MAX_EPOCHS = 10
PATIENCE = None  # None: obliga a intentar las 10 épocas completas
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.30
NUM_WORKERS = 4
VAL_THRESHOLD = 0.5
USE_AMP = True
CONTINUE_AFTER_ERROR = True
SAVE_BEST_MODEL = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = NOTEBOOK_OUTPUT_ROOT / f"run_oom_bs8_10ep_{RUN_TAG}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("RUN_ROOT:", RUN_ROOT)
print("DEVICE:", DEVICE)
print("N.º preprocesados:", len(PREPROCESSINGS))
print("N.º experimentos:", len(EXPERIMENTS))
print("N.º total de entrenamientos:", len(PREPROCESSINGS) * len(EXPERIMENTS))


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)

In [ ]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def split_complications(raw_value: str):
    value = normalize_text(raw_value)
    if value == "" or value.lower() == "x":
        return []
    parts = [p.strip() for p in value.split(",")]
    return [p for p in parts if p]

def build_multilabel_dataframe(clinical_csv: Path):
    df = pd.read_csv(clinical_csv)
    df = df.copy()
    df["patient_id"] = df["patient_id"].astype(str)
    df["Complicación"] = df["Complicación"].apply(normalize_text)
    df["Tipo de complicación"] = df["Tipo de complicación"].apply(normalize_text)

    parsed = df["Tipo de complicación"].apply(split_complications)
    df["Hemorragia"] = parsed.apply(lambda xs: int("Hemorragia" in xs))
    df["Neumotórax"] = parsed.apply(lambda xs: int("Neumotórax" in xs))
    df["Derrame_pleural"] = parsed.apply(lambda xs: int("Derrame pleural" in xs))

    off_target_only = (
        (df["Hemorragia"] == 0)
        & (df["Neumotórax"] == 0)
        & (df["Derrame_pleural"] == 1)
    )

    if DROP_OFF_TARGET_ONLY:
        df = df.loc[~off_target_only].copy()

    df["Sin_complicacion"] = ((df["Hemorragia"] == 0) & (df["Neumotórax"] == 0)).astype(int)

    label_cols = ["Hemorragia", "Neumotórax", "Sin_complicacion"]
    df["labelset_key"] = df[label_cols].astype(str).agg("".join, axis=1)
    return df

labels_df = build_multilabel_dataframe(CLINICAL_CSV)
print("Pacientes tras preparar endpoint:", len(labels_df))
display(labels_df[["patient_id", "Complicación", "Tipo de complicación", "Hemorragia", "Neumotórax", "Sin_complicacion"]].head())
print(labels_df[["Hemorragia", "Neumotórax", "Sin_complicacion"]].sum())
print(labels_df["labelset_key"].value_counts())

In [ ]:
MASK_DEPENDENCIES = {
    "lung": "masks_lung",
    "nodule": "masks_nodule",
    "vessels": "masks_vessels",
    "trachea_bronchia": "masks_trachea_bronchia",
}

CHANNEL_REQUIREMENTS = {
    "ct_lung": ["lung"],
    "ct_nodule": ["nodule"],
    "lung": ["lung"],
    "nodule": ["nodule"],
    "vessels": ["vessels"],
    "trachea_bronchia": ["trachea_bronchia"],
    "weighted_ct": ["lung", "nodule", "vessels"],
}

def required_masks_for_experiment(exp_cfg):
    needed = set()
    for channel in exp_cfg["channels"]:
        needed.update(CHANNEL_REQUIREMENTS[channel])
    return sorted(needed)

def preproc_patient_ids(preproc_name, mask_keys):
    base = PREPROC_ROOT / preproc_name / "npy"
    images_dir = base / "images"
    image_ids = {p.stem for p in images_dir.glob("*.npy")}
    current = set(image_ids)
    for mask_key in mask_keys:
        mask_dir = base / MASK_DEPENDENCIES[mask_key]
        mask_ids = {p.stem for p in mask_dir.glob("*.npy")}
        current &= mask_ids
    return current

def common_ids_for_selection(preproc_names, experiments, labels_df):
    label_ids = set(labels_df["patient_id"].astype(str))
    if not preproc_names or not experiments:
        return sorted(label_ids)
    available = None
    for prep in preproc_names:
        for exp in experiments:
            pids = preproc_patient_ids(prep, required_masks_for_experiment(exp))
            if available is None:
                available = set(pids)
            elif REQUIRE_COMMON_COHORT:
                available &= set(pids)
            else:
                available |= set(pids)
    return sorted(label_ids & (available if available is not None else set()))

selected_preprocs = PREPROCESSINGS[:MAX_PREPROCS] if MAX_PREPROCS else PREPROCESSINGS
selected_experiments = EXPERIMENTS[:MAX_EXPERIMENTS] if MAX_EXPERIMENTS else EXPERIMENTS
common_ids = common_ids_for_selection(selected_preprocs, selected_experiments, labels_df)
print("Preprocesados seleccionados:", selected_preprocs)
print("Experimentos seleccionados:", [e["name"] for e in selected_experiments])
print("Tamano de cohorte comun:", len(common_ids))
labels_df = labels_df[labels_df["patient_id"].isin(common_ids)].copy().reset_index(drop=True)
print(labels_df[["Hemorragia", "Neumotórax", "Sin_complicacion"]].sum())
print(labels_df["labelset_key"].value_counts())

In [ ]:
def pick_example_shape(preproc_name):
    img_dir = PREPROC_ROOT / preproc_name / "npy" / "images"
    first = next(img_dir.glob("*.npy"))
    arr = np.load(first, mmap_mode="r")
    return first.stem, arr.shape, arr.dtype

shape_rows = []
for prep in selected_preprocs:
    pid, shape, dtype = pick_example_shape(prep)
    shape_rows.append({
        "preproc": prep,
        "example_patient": pid,
        "raw_shape": str(shape),
        "dtype": str(dtype),
    })
display(pd.DataFrame(shape_rows))


In [ ]:
def make_transforms(train=True):
    if train:
        return Compose([
            RandFlip(prob=0.5, spatial_axis=0),
            RandFlip(prob=0.5, spatial_axis=1),
            RandFlip(prob=0.5, spatial_axis=2),
            RandRotate90(prob=0.5, max_k=3),
            RandAffine(
                prob=0.20,
                translate_range=(4, 4, 4),
                scale_range=(0.05, 0.05, 0.05),
                rotate_range=(0.10, 0.10, 0.10),
                padding_mode="zeros",
            ),
            EnsureType(),
        ])
    return Compose([EnsureType()])


class MultiLabelMaskedVolumeDataset(Dataset):
    def __init__(self, labels_frame, patient_ids, preproc_name, experiment_cfg, transform=None):
        self.df = labels_frame.set_index("patient_id")
        self.patient_ids = list(patient_ids)
        self.preproc_name = preproc_name
        self.experiment_cfg = experiment_cfg
        self.transform = transform
        self.base_dir = PREPROC_ROOT / preproc_name / "npy"
        self.image_dir = self.base_dir / "images"
        self.mask_dirs = {k: self.base_dir / v for k, v in MASK_DEPENDENCIES.items()}
        self.label_cols = TARGET_LABELS

    def __len__(self):
        return len(self.patient_ids)

    @staticmethod
    def _as_channel_first_image(arr, path):
        # Resultado esperado: [C, D, H, W].
        arr = np.asarray(arr)
        arr = np.squeeze(arr)

        if arr.ndim == 3:
            return arr[None, ...]

        if arr.ndim != 4:
            raise ValueError(f"Imagen con dimensionalidad no soportada: {path} -> {arr.shape}")

        # Multiwindowing puede guardarse como [C,D,H,W] o [D,H,W,C].
        first_is_channel = arr.shape[0] <= 8 and all(s > 8 for s in arr.shape[1:])
        last_is_channel = arr.shape[-1] <= 8 and all(s > 8 for s in arr.shape[:-1])

        if first_is_channel and not last_is_channel:
            return arr
        if last_is_channel and not first_is_channel:
            return np.moveaxis(arr, -1, 0)
        if first_is_channel and last_is_channel:
            # Caso ambiguo poco frecuente: se prioriza canal primero.
            return arr

        raise ValueError(
            f"No se pudo localizar el eje de canales de la imagen 4D: {path} -> {arr.shape}"
        )

    @staticmethod
    def _as_3d_mask(arr, path):
        arr = np.asarray(arr)
        arr = np.squeeze(arr)
        if arr.ndim == 3:
            return arr

        if arr.ndim == 4:
            # Admite máscaras con un único canal explícito.
            if arr.shape[0] == 1:
                return arr[0]
            if arr.shape[-1] == 1:
                return arr[..., 0]

        raise ValueError(f"Máscara con dimensionalidad no soportada: {path} -> {arr.shape}")

    def _load_image(self, path: Path):
        arr = np.load(path, mmap_mode="r")
        arr = self._as_channel_first_image(arr, path)
        return arr.astype(np.float32, copy=False)

    def _load_mask(self, path: Path):
        arr = np.load(path, mmap_mode="r")
        arr = self._as_3d_mask(arr, path)
        return (arr > 0).astype(np.float32, copy=False)

    @staticmethod
    def _mask_image(img, mask):
        # img: [C,D,H,W], mask: [D,H,W]
        if tuple(img.shape[1:]) != tuple(mask.shape):
            raise ValueError(f"Shape incompatible imagen/máscara: {img.shape} vs {mask.shape}")
        return img * mask[None, ...]

    def _build_channels(self, img, masks):
        lung = masks.get("lung")
        nodule = masks.get("nodule")
        vessels = masks.get("vessels")
        trb = masks.get("trachea_bronchia")

        channel_map = {
            "ct_lung": self._mask_image(img, lung) if lung is not None else img,
            "ct_nodule": self._mask_image(img, nodule) if nodule is not None else img,
            "lung": lung[None, ...] if lung is not None else None,
            "nodule": nodule[None, ...] if nodule is not None else None,
            "vessels": vessels[None, ...] if vessels is not None else None,
            "trachea_bronchia": trb[None, ...] if trb is not None else None,
        }

        if "weighted_ct" in self.experiment_cfg["channels"]:
            weights = self.experiment_cfg.get(
                "weights",
                {"lung": 0.25, "nodule": 1.0, "vessels": 0.75},
            )
            weight_map = np.zeros(img.shape[1:], dtype=np.float32)
            if lung is not None:
                weight_map += weights.get("lung", 0.0) * lung
            if nodule is not None:
                weight_map += weights.get("nodule", 0.0) * nodule
            if vessels is not None:
                weight_map += weights.get("vessels", 0.0) * vessels
            weight_map = np.clip(weight_map, 0.0, 1.5)
            channel_map["weighted_ct"] = img * weight_map[None, ...]

        selected = []
        for name in self.experiment_cfg["channels"]:
            value = channel_map[name]
            if value is None:
                raise ValueError(f"Canal requerido no disponible: {name}")
            selected.append(value)

        # Cada elemento ya tiene forma [C,D,H,W]; se concatenan los canales.
        channels = np.concatenate(selected, axis=0).astype(np.float32, copy=False)
        channels = np.nan_to_num(channels, nan=0.0, posinf=0.0, neginf=0.0)
        channels = np.clip(channels, 0.0, 1.0)
        return channels

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        img = self._load_image(self.image_dir / f"{pid}.npy")

        masks = {}
        for mask_key in required_masks_for_experiment(self.experiment_cfg):
            masks[mask_key] = self._load_mask(
                self.mask_dirs[mask_key] / f"{pid}.npy"
            )

        x = self._build_channels(img, masks)
        if self.transform is not None:
            x = self.transform(x)

        y = self.df.loc[pid, self.label_cols].values.astype(np.float32)
        y = torch.tensor(y, dtype=torch.float32)
        return x, y, pid


In [ ]:
def stratified_split_ids(labels_frame, train_size=0.8, seed=42):
    keys = labels_frame["labelset_key"]
    train_ids, val_ids = train_test_split(
        labels_frame["patient_id"].tolist(),
        train_size=train_size,
        random_state=seed,
        stratify=keys,
    )
    return list(train_ids), list(val_ids)

train_ids, val_ids = stratified_split_ids(labels_df, train_size=TRAIN_SIZE, seed=RANDOM_SEED)
print("Train:", len(train_ids), "Val:", len(val_ids))
print("Distribucion train")
display(labels_df[labels_df["patient_id"].isin(train_ids)][TARGET_LABELS].sum().to_frame("train"))
print("Distribucion val")
display(labels_df[labels_df["patient_id"].isin(val_ids)][TARGET_LABELS].sum().to_frame("val"))

In [ ]:
def build_model(in_channels, n_outputs, dropout=DROPOUT):
    model = resnet18(
        spatial_dims=3,
        n_input_channels=in_channels,
        num_classes=n_outputs,
    )
    if dropout and dropout > 0:
        model.fc = nn.Sequential(nn.Dropout(dropout), model.fc)
    return model

def compute_pos_weight(labels_frame, patient_ids):
    y = labels_frame.set_index("patient_id").loc[patient_ids, TARGET_LABELS].values.astype(np.float32)
    pos = y.sum(axis=0)
    neg = len(patient_ids) - pos
    pos_weight = np.where(pos > 0, neg / np.maximum(pos, 1.0), 1.0)
    return torch.tensor(pos_weight, dtype=torch.float32)

def sigmoid_probs(logits):
    return torch.sigmoid(logits).detach().cpu().numpy()

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_micro": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "subset_accuracy": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "jaccard_micro": jaccard_score(y_true, y_pred, average="micro", zero_division=0),
    }
    for idx, label in enumerate(TARGET_LABELS):
        metrics[f"f1_{label}"] = f1_score(y_true[:, idx], y_pred[:, idx], zero_division=0)
    return metrics

@torch.no_grad()
def evaluate_model(model, loader, threshold=0.5):
    model.eval()
    all_logits = []
    all_targets = []
    all_ids = []
    total_loss = 0.0
    batches = 0
    for xb, yb, pids in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        logits = model(xb)
        all_logits.append(logits.detach().cpu())
        all_targets.append(yb.detach().cpu())
        all_ids.extend(pids)
        batches += 1
    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_targets).numpy()
    y_prob = 1.0 / (1.0 + np.exp(-logits))
    metrics = compute_metrics(y_true, y_prob, threshold=threshold)
    pred_df = pd.DataFrame(y_prob, columns=[f"prob_{c}" for c in TARGET_LABELS])
    pred_df.insert(0, "patient_id", all_ids)
    for idx, label in enumerate(TARGET_LABELS):
        pred_df[f"true_{label}"] = y_true[:, idx].astype(int)
        pred_df[f"pred_{label}"] = (y_prob[:, idx] >= threshold).astype(int)
    return metrics, pred_df

In [ ]:
def plot_history(history_df, title, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Entrenamiento")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validacion")
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("Perdida")
    axes[0].set_title("Evolucion de la perdida")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["train_f1_micro"], label="Train F1 micro")
    axes[1].plot(history_df["epoch"], history_df["val_f1_micro"], label="Val F1 micro")
    axes[1].plot(history_df["epoch"], history_df["val_f1_macro"], label="Val F1 macro")
    axes[1].plot(history_df["epoch"], history_df["val_subset_accuracy"], label="Val subset acc")
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("Valor")
    axes[1].set_title("Evolucion de metricas")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
def is_cuda_oom(exc):
    oom_type = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)
    return isinstance(exc, oom_type) or "out of memory" in str(exc).lower()


def cuda_memory_gib():
    if not torch.cuda.is_available():
        return {
            "gpu_allocated_gib": 0.0,
            "gpu_reserved_gib": 0.0,
            "gpu_peak_allocated_gib": 0.0,
            "gpu_peak_reserved_gib": 0.0,
            "gpu_total_gib": 0.0,
        }
    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    return {
        "gpu_allocated_gib": torch.cuda.memory_allocated(device_idx) / 1024**3,
        "gpu_reserved_gib": torch.cuda.memory_reserved(device_idx) / 1024**3,
        "gpu_peak_allocated_gib": torch.cuda.max_memory_allocated(device_idx) / 1024**3,
        "gpu_peak_reserved_gib": torch.cuda.max_memory_reserved(device_idx) / 1024**3,
        "gpu_total_gib": props.total_memory / 1024**3,
    }


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def make_loader(dataset, shuffle):
    kwargs = dict(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
    )
    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = 1
    return DataLoader(**kwargs)


def run_single_experiment(preproc_name, experiment_cfg):
    exp_name = experiment_cfg["name"]
    exp_dir = RUN_ROOT / f"{preproc_name}__{exp_name}"
    exp_dir.mkdir(parents=True, exist_ok=True)

    current_stage = "initialization"
    current_epoch = 0
    current_batch = 0
    started_at = time.time()

    status = "OK"
    error_type = None
    error_message = None
    error_traceback = None

    best_score = -np.inf
    best_epoch = None
    best_state = None
    history = []
    in_channels = None
    train_ds = val_ds = train_loader = val_loader = None
    model = criterion = optimizer = scheduler = scaler = None

    cleanup_cuda()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    try:
        current_stage = "dataset"
        train_ds = MultiLabelMaskedVolumeDataset(
            labels_frame=labels_df,
            patient_ids=train_ids,
            preproc_name=preproc_name,
            experiment_cfg=experiment_cfg,
            transform=make_transforms(train=True),
        )
        val_ds = MultiLabelMaskedVolumeDataset(
            labels_frame=labels_df,
            patient_ids=val_ids,
            preproc_name=preproc_name,
            experiment_cfg=experiment_cfg,
            transform=make_transforms(train=False),
        )

        current_stage = "dataloader"
        train_loader = make_loader(train_ds, shuffle=True)
        val_loader = make_loader(val_ds, shuffle=False)

        current_stage = "sample_loading"
        sample_x, _, _ = train_ds[0]
        in_channels = int(sample_x.shape[0])
        print(
            f"\n[{preproc_name} | {exp_name}] "
            f"sample={tuple(sample_x.shape)} in_channels={in_channels}"
        )

        current_stage = "model_to_device"
        model = build_model(
            in_channels=in_channels,
            n_outputs=len(TARGET_LABELS),
            dropout=DROPOUT,
        ).to(DEVICE)

        pos_weight = compute_pos_weight(labels_df, train_ids).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=3,
        )
        scaler = GradScaler(enabled=(USE_AMP and DEVICE.type == "cuda"))

        patience_counter = 0

        for epoch in range(1, MAX_EPOCHS + 1):
            current_epoch = epoch
            current_stage = "train"
            model.train()
            epoch_losses = []
            train_probs = []
            train_targets = []

            for batch_idx, (xb, yb, _) in enumerate(train_loader, start=1):
                current_batch = batch_idx
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with autocast(enabled=(USE_AMP and DEVICE.type == "cuda")):
                    logits = model(xb)
                    loss = criterion(logits, yb)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                epoch_losses.append(loss.item())
                train_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
                train_targets.append(yb.detach().cpu().numpy())

                # Libera referencias al lote antes del siguiente.
                del xb, yb, logits, loss

            train_loss = float(np.mean(epoch_losses)) if epoch_losses else np.nan
            train_probs = np.concatenate(train_probs, axis=0)
            train_targets = np.concatenate(train_targets, axis=0)
            train_metrics = compute_metrics(
                train_targets,
                train_probs,
                threshold=VAL_THRESHOLD,
            )

            current_stage = "validation"
            current_batch = 0
            model.eval()
            val_losses = []
            val_probs = []
            val_targets = []
            val_ids_local = []

            with torch.no_grad():
                for batch_idx, (xb, yb, pids) in enumerate(val_loader, start=1):
                    current_batch = batch_idx
                    xb = xb.to(DEVICE, non_blocking=True)
                    yb = yb.to(DEVICE, non_blocking=True)

                    with autocast(enabled=(USE_AMP and DEVICE.type == "cuda")):
                        logits = model(xb)
                        loss = criterion(logits, yb)

                    val_losses.append(loss.item())
                    val_probs.append(torch.sigmoid(logits).cpu().numpy())
                    val_targets.append(yb.cpu().numpy())
                    val_ids_local.extend(pids)
                    del xb, yb, logits, loss

            val_loss = float(np.mean(val_losses)) if val_losses else np.nan
            val_probs = np.concatenate(val_probs, axis=0)
            val_targets = np.concatenate(val_targets, axis=0)
            val_metrics = compute_metrics(
                val_targets,
                val_probs,
                threshold=VAL_THRESHOLD,
            )
            scheduler.step(val_loss)

            mem = cuda_memory_gib()
            epoch_row = {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "lr": optimizer.param_groups[0]["lr"],
                "train_f1_micro": train_metrics["f1_micro"],
                "train_f1_macro": train_metrics["f1_macro"],
                "val_f1_micro": val_metrics["f1_micro"],
                "val_f1_macro": val_metrics["f1_macro"],
                "val_subset_accuracy": val_metrics["subset_accuracy"],
                "val_hamming_loss": val_metrics["hamming_loss"],
                "val_precision_micro": val_metrics["precision_micro"],
                "val_recall_micro": val_metrics["recall_micro"],
                "val_jaccard_micro": val_metrics["jaccard_micro"],
                "gpu_allocated_gib": mem["gpu_allocated_gib"],
                "gpu_reserved_gib": mem["gpu_reserved_gib"],
                "gpu_peak_allocated_gib": mem["gpu_peak_allocated_gib"],
                "gpu_peak_reserved_gib": mem["gpu_peak_reserved_gib"],
            }
            for label in TARGET_LABELS:
                epoch_row[f"val_f1_{label}"] = val_metrics[f"f1_{label}"]
            history.append(epoch_row)

            score = val_metrics["f1_micro"]
            improved = score > best_score + 1e-4
            if improved:
                best_score = score
                best_epoch = epoch
                patience_counter = 0

                if SAVE_BEST_MODEL:
                    current_stage = "save_best_model"
                    best_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()
                    }
                    torch.save(best_state, exp_dir / "best_model.pt")

                best_pred_df = pd.DataFrame(
                    val_probs,
                    columns=[f"prob_{c}" for c in TARGET_LABELS],
                )
                best_pred_df.insert(0, "patient_id", val_ids_local)
                for idx, label in enumerate(TARGET_LABELS):
                    best_pred_df[f"true_{label}"] = val_targets[:, idx].astype(int)
                    best_pred_df[f"pred_{label}"] = (
                        val_probs[:, idx] >= VAL_THRESHOLD
                    ).astype(int)
                best_pred_df.to_csv(
                    exp_dir / "val_predictions_best_epoch.csv",
                    index=False,
                )
            else:
                patience_counter += 1

            print(
                f"[{preproc_name} | {exp_name}] "
                f"epoch={epoch:02d}/{MAX_EPOCHS} "
                f"train_loss={train_loss:.4f} "
                f"val_loss={val_loss:.4f} "
                f"val_f1_micro={val_metrics['f1_micro']:.4f} "
                f"peak_alloc={mem['gpu_peak_allocated_gib']:.2f} GiB "
                f"peak_reserved={mem['gpu_peak_reserved_gib']:.2f} GiB"
            )

            # En esta prueba PATIENCE=None para intentar siempre 10 épocas.
            if PATIENCE is not None and patience_counter >= PATIENCE:
                print(
                    f"Early stopping en época {epoch} para "
                    f"{preproc_name} | {exp_name}"
                )
                break

        current_stage = "completed"

    except Exception as exc:
        if is_cuda_oom(exc):
            status = "OOM"
        else:
            status = "ERROR"

        error_type = type(exc).__name__
        error_message = str(exc)
        error_traceback = traceback.format_exc()
        print(
            f"\n[{status}] {preproc_name} | {exp_name} "
            f"stage={current_stage} epoch={current_epoch} batch={current_batch}\n"
            f"{error_type}: {error_message}"
        )

        with open(exp_dir / "error_traceback.txt", "w", encoding="utf-8") as f:
            f.write(error_traceback)

    finally:
        elapsed_seconds = time.time() - started_at
        mem = cuda_memory_gib()

        history_df = pd.DataFrame(history)
        history_df.to_csv(exp_dir / "history.csv", index=False)

        if not history_df.empty:
            try:
                plot_history(
                    history_df,
                    title=f"{preproc_name} | {exp_name} | {status}",
                    save_path=exp_dir / "curvas.png",
                )
            except Exception as plot_exc:
                print("No se pudo crear la gráfica:", plot_exc)

        summary = {
            "preprocessing": preproc_name,
            "experiment": exp_name,
            "description": experiment_cfg.get("description", ""),
            "channels": "|".join(experiment_cfg["channels"]),
            "weights": json.dumps(experiment_cfg.get("weights"), ensure_ascii=False),
            "status": status,
            "batch_size": BATCH_SIZE,
            "max_epochs_requested": MAX_EPOCHS,
            "epochs_completed": len(history),
            "failure_stage": None if status == "OK" else current_stage,
            "failure_epoch": None if status == "OK" else current_epoch,
            "failure_batch": None if status == "OK" else current_batch,
            "error_type": error_type,
            "error_message": error_message,
            "in_channels": in_channels,
            "train_size": len(train_ids),
            "val_size": len(val_ids),
            "best_epoch": int(best_epoch) if best_epoch is not None else None,
            "best_val_f1_micro": (
                float(best_score) if best_epoch is not None else None
            ),
            "elapsed_seconds": float(elapsed_seconds),
            **mem,
        }

        if best_epoch is not None and not history_df.empty:
            best_row = (
                history_df.loc[history_df["epoch"] == best_epoch]
                .iloc[0]
                .to_dict()
            )
            for key, value in best_row.items():
                if isinstance(value, (np.floating, float)):
                    summary[key] = float(value)
                elif isinstance(value, (np.integer, int)):
                    summary[key] = int(value)
                else:
                    summary[key] = value

        with open(exp_dir / "summary.json", "w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

        # Elimina referencias antes de pasar a la siguiente combinación.
        best_state = None
        model = criterion = optimizer = scheduler = scaler = None
        train_loader = val_loader = train_ds = val_ds = None
        cleanup_cuda()

    return summary, history_df


In [ ]:
all_summaries = []
all_histories = {}

progress_csv = RUN_ROOT / "summary_all_experiments.csv"

for prep_idx, prep in enumerate(selected_preprocs, start=1):
    for exp_idx, exp_cfg in enumerate(selected_experiments, start=1):
        print(
            f"\n===== Preprocesado {prep_idx}/{len(selected_preprocs)} | "
            f"Experimento {exp_idx}/{len(selected_experiments)} ====="
        )

        summary, history_df = run_single_experiment(prep, exp_cfg)
        all_summaries.append(summary)
        if not history_df.empty:
            all_histories[(prep, exp_cfg["name"])] = history_df

        # Checkpoint incremental: si el proceso se interrumpe, se conserva lo ya probado.
        pd.DataFrame(all_summaries).to_csv(progress_csv, index=False)

        if summary["status"] == "ERROR" and not CONTINUE_AFTER_ERROR:
            raise RuntimeError(
                f"Se detuvo el barrido por ERROR en {prep} | {exp_cfg['name']}"
            )

summary_df = pd.DataFrame(all_summaries)

status_order = pd.CategoricalDtype(
    categories=["OOM", "ERROR", "OK"],
    ordered=True,
)
summary_df["status_sort"] = summary_df["status"].astype(status_order)
summary_df = (
    summary_df
    .sort_values(
        ["status_sort", "preprocessing", "experiment"],
        ascending=[True, True, True],
    )
    .drop(columns="status_sort")
    .reset_index(drop=True)
)
summary_df.to_csv(RUN_ROOT / "summary_all_experiments.csv", index=False)

display_cols = [
    "preprocessing",
    "experiment",
    "status",
    "in_channels",
    "epochs_completed",
    "failure_stage",
    "failure_epoch",
    "failure_batch",
    "gpu_peak_allocated_gib",
    "gpu_peak_reserved_gib",
    "elapsed_seconds",
]
display(summary_df[display_cols])


In [ ]:
# Matriz de estado por preprocesado y experimento
oom_matrix = summary_df.pivot(
    index="preprocessing",
    columns="experiment",
    values="status",
)
oom_matrix.to_csv(RUN_ROOT / "oom_matrix.csv")
display(oom_matrix)

# Resumen agregado por preprocesado
status_counts = (
    summary_df.groupby(["preprocessing", "status"])
    .size()
    .unstack(fill_value=0)
)
for col in ["OK", "OOM", "ERROR"]:
    if col not in status_counts.columns:
        status_counts[col] = 0

status_counts = status_counts[["OK", "OOM", "ERROR"]].rename(
    columns={"OK": "n_ok", "OOM": "n_oom", "ERROR": "n_error"}
)
status_counts["n_runs"] = status_counts[["n_ok", "n_oom", "n_error"]].sum(axis=1)
status_counts["all_ok"] = status_counts["n_ok"] == status_counts["n_runs"]
status_counts["any_oom"] = status_counts["n_oom"] > 0

oom_runs = (
    summary_df[summary_df["status"] == "OOM"]
    .groupby("preprocessing")["experiment"]
    .apply(lambda x: " | ".join(x))
)
error_runs = (
    summary_df[summary_df["status"] == "ERROR"]
    .groupby("preprocessing")["experiment"]
    .apply(lambda x: " | ".join(x))
)

summary_by_preprocessing = status_counts.copy()
summary_by_preprocessing["oom_experiments"] = oom_runs
summary_by_preprocessing["error_experiments"] = error_runs
summary_by_preprocessing = summary_by_preprocessing.reset_index()
summary_by_preprocessing.to_csv(
    RUN_ROOT / "summary_by_preprocessing.csv",
    index=False,
)

display(summary_by_preprocessing)

print("\nPreprocesados con al menos un OOM:")
oom_preps = summary_by_preprocessing.loc[
    summary_by_preprocessing["any_oom"],
    ["preprocessing", "n_oom", "oom_experiments"],
]
display(oom_preps)

print("\nPreprocesados que completaron todos los experimentos sin OOM ni errores:")
ok_preps = summary_by_preprocessing.loc[
    summary_by_preprocessing["all_ok"],
    ["preprocessing", "n_ok"],
]
display(ok_preps)


## Cómo interpretar el resultado

- `OK`: la combinación completó las 10 épocas con `batch_size=8`.
- `OOM`: PyTorch detectó falta de memoria CUDA; se registra la etapa, época y lote.
- `ERROR`: ocurrió un fallo distinto de OOM, por ejemplo un archivo ausente o una forma incompatible.

La columna `any_oom` de `summary_by_preprocessing.csv` indica si un preprocesado produjo OOM en al menos una configuración de canales. La matriz `oom_matrix.csv` permite ver exactamente qué configuración lo provocó.

El barrido continúa después de cada OOM y limpia la caché CUDA antes de empezar la siguiente combinación. Además, `summary_all_experiments.csv` se actualiza después de cada entrenamiento para no perder resultados si el proceso se interrumpe.

Como se prueban 96 entrenamientos, el tiempo total puede ser elevado. Para una prueba inicial rápida se puede limitar temporalmente `MAX_EXPERIMENTS`, pero la configuración entregada ejecuta todos los preprocesados y todos los experimentos.
